In [1]:
import sys
sys.path.insert(0, '/workspace')

# Register config_local as 'config' so all submodules (datasets, models, etc.) use local settings
import config_local
sys.modules['config'] = config_local
import config_local as config

import torch
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as T
from experiments.neuralnet.models import AE, Decoder, BetaVAE, BetaVAEScalingLLM #models_resnet import BetaVAE, AE
from experiments.neuralnet.datasets import CocoTextEmbeddingImageDataset
from torch.utils.data import DataLoader

# Model set-up

In [2]:
last_epoch = 50
#checkpoint_path = f"/workspace/data/checkpoints/vanilla_resnet18_ae_loss_l2/cocoDoerig/run_0/checkpoint_epoch{last_epoch}.ckpt"
#checkpoint_path = f"/workspace/data/checkpoints/vanilla_resnet18_ae_loss_l2/cocoDoerig/run_0/checkpoint_epoch{last_epoch}.ckpt"
#checkpoint_path = f"/workspace/data/checkpoints/vanilla_from_ae_beta_vae_loss_standard_beta0.001_recon_loss_l2/cocoDoerig/run_0/checkpoint_epoch{last_epoch}.ckpt"
checkpoint_path = f"/workspace/data/checkpoints/beta_vae_llm_loss_l2_and_img_norm_delta0.1/cocoDoerig/run_0/checkpoint_epoch{last_epoch}.ckpt"

model_type = "beta_vae_llm"

if model_type == "decoder":
    model = Decoder(
        latent_dim=config.latent_dim,
        image_size=config.img_resize,
        checkpoint_path=checkpoint_path,
    )
elif model_type == "ae":
    model = AE(
        latent_dim=config.latent_dim,
        image_size=config.img_resize,
        checkpoint_path=checkpoint_path,
        encoder_checkpoint=False,
        )
    
elif model_type == "beta_vae":
    model = BetaVAE(
        latent_dim=config.latent_dim,
        image_size=config.img_resize,
        checkpoint_path=checkpoint_path,
        ae_checkpoint=False,
    )
elif model_type == "beta_vae_llm":
    model = BetaVAEScalingLLM(
        latent_dim=config.latent_dim,
        image_size=config.img_resize,
        checkpoint_path=checkpoint_path,
        vae_checkpoint=False,
    )

model.eval()
print('ready')
#print(f"Model loaded. latent_dim={model.latent_dim}, image_size={model.image_size}")


Weights loaded from checkpoint: /workspace/data/checkpoints/beta_vae_llm_loss_l2_and_img_norm_delta0.1/cocoDoerig/run_0/checkpoint_epoch50.ckpt
ready


# Image encoding/decoding

### Helper: denormalize & display

In [3]:
img_mean_t = torch.tensor(config.img_mean).view(1, 3, 1, 1)
img_std_t  = torch.tensor(config.img_std).view(1, 3, 1, 1)

def denormalize(t):
    return (t.cpu() * img_std_t + img_mean_t).clamp(0, 1)

def show_comparison(originals, reconstructed, title="Original vs Reconstructed"):
    n = originals.shape[0]
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    for i in range(n):
        orig_img = denormalize(originals[i:i+1]).squeeze(0).permute(1, 2, 0).numpy()
        recon_img = denormalize(reconstructed[i:i+1]).squeeze(0).permute(1, 2, 0).detach().numpy()
        axes[0, i].imshow(orig_img)
        axes[0, i].set_title(f'Original {i}')
        axes[0, i].axis('off')
        axes[1, i].imshow(recon_img)
        axes[1, i].set_title(f'Reconstructed {i}')
        axes[1, i].axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

### 1. artificial images

In [4]:
def create_synthetic_images(n=4, image_size=224):
    images = []

    # Pattern 1: Random noise
    images.append(torch.rand(3, image_size, image_size))

    # Pattern 2: Gradient
    x = torch.linspace(0, 1, image_size)
    y = torch.linspace(0, 1, image_size)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    images.append(torch.stack([X, Y, (X + Y) / 2], dim=0))

    # Pattern 3: Circle
    cx = image_size // 2
    xi = torch.arange(image_size).float() - cx
    yi = torch.arange(image_size).float() - cx
    Xi, Yi = torch.meshgrid(xi, yi, indexing='ij')
    circle = torch.exp(-(Xi**2 + Yi**2) / (image_size / 4)**2)
    images.append(torch.stack([circle, circle * 0.5, 1 - circle], dim=0))

    # Pattern 4: Checkerboard
    checker = ((Xi // 32 + Yi // 32) % 2).float()
    images.append(torch.stack([checker, 1 - checker, checker * 0.5], dim=0))

    imgs = torch.stack(images[:n], dim=0)  # [0, 1] range
    normalize = T.Normalize(mean=config.img_mean, std=config.img_std)
    imgs = torch.stack([normalize(img) for img in imgs])
    return imgs

if model_type in ["decoder", "beta_vae_llm"]:
    pass
else:
    synthetic = create_synthetic_images(4, config.img_resize)
    print(f"Synthetic input shape: {synthetic.shape}")

    with torch.inference_mode():
        if model_type == "beta_vae":
            img_hat_syn, mu, log_var, latent_syn = model(synthetic)
        else:
            img_hat_syn, latent_syn = model(synthetic)

    show_comparison(synthetic, img_hat_syn, "Synthetic Images: Original vs Reconstructed")

### 2. coco datasets

In [5]:
dataset = CocoTextEmbeddingImageDataset(
    split="val",
    img_transform=config.img_transform_val,
)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)
batch = next(iter(dataloader))
images = batch["image"]
text_embeddings = batch["text_embedding"]
print(f"Images shape: {images.shape}")
print(f"Text embeddings shape: {text_embeddings.shape}")

with torch.inference_mode():
    if model_type == "beta_vae":
        img_hat_coco, mu_coco, log_var_coco, latent_coco = model(images)
    elif model_type == "ae":
        img_hat_coco, latent_coco = model(images)
    elif model_type == "decoder":
        img_hat_coco = model(text_embeddings)
    elif model_type == "beta_vae_llm":
        img_hat_coco, scaling_factor = model(text_embeddings)
        intermediate = model.encoder(images)
        intermediate = torch.flatten(intermediate, start_dim=1)
        mu_coco = model.fc_mu(intermediate)
        log_var_coco = model.fc_logvar(intermediate)
show_comparison(images[:5], img_hat_coco[:5], "COCO Images: Original vs Reconstructed")

NotImplementedError: This dataset is deprecated. Please use CocoH5Dataset instead.

In [ ]:
if model_type == "beta_vae_llm":
    print(f"Scaling factor from LLM: {scaling_factor.T}")
    print(f"Norm text embeddings after scaling: {(text_embeddings * scaling_factor).norm(p=2, dim=1)}")
    print(f"Norm latent representation (mu): {mu_coco.norm(p=2, dim=1)}")

Scaling factor from LLM: tensor([[-6.4667, -7.1111, -4.8414, -7.1515, -5.6239, -5.2612, -6.4715, -6.1161]])
Norm text embeddings after scaling: tensor([6.4667, 7.1111, 4.8414, 7.1515, 5.6239, 5.2612, 6.4715, 6.1161])
Norm latent representation (mu): tensor([4.0122, 3.1929, 2.9325, 4.3418, 3.5253, 6.1379, 3.6385, 4.3955])


In [ ]:
# Active dimensions analysis
with torch.inference_mode():
    # Collect mu and logvar over more batches for stable estimate
    all_mu = []
    all_logvar = []
    for i, batch in enumerate(dataloader):
        imgs = batch["image"]
        _, mu_b, logvar_b, _ = model(imgs)
        all_mu.append(mu_b)
        all_logvar.append(logvar_b)
        if i >= 9:  # 10 batches
            break

all_mu = torch.cat(all_mu, dim=0)
all_logvar = torch.cat(all_logvar, dim=0)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (3072x128 and 768x1)

In [ ]:
# mu values per dimension (sorted by mean)
# all_mu: [N, latent_dim]
mu_mean = all_mu.mean(0).cpu().numpy()
mu_std  = all_mu.std(0).cpu().numpy()

sort_idx = mu_mean.argsort()
mu_mean_sorted = mu_mean[sort_idx]
mu_std_sorted  = mu_std[sort_idx]

dims = range(len(mu_mean_sorted))

plt.figure(figsize=(14, 4))
plt.plot(dims, mu_mean_sorted, linewidth=0.8, label='mean')
plt.fill_between(dims, mu_mean_sorted - mu_std_sorted, mu_mean_sorted + mu_std_sorted, alpha=0.3, label='±1 std')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel("Rank (sorted by mean mu)")
plt.ylabel("mu")
plt.title("mu per latent dimension — sorted by mean")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# logvar values per dimension (sorted by mean)
logvar_mean = all_logvar.mean(0).cpu().numpy()
logvar_std  = all_logvar.std(0).cpu().numpy()

sort_idx = logvar_mean.argsort()
logvar_mean_sorted = logvar_mean[sort_idx]
logvar_std_sorted  = logvar_std[sort_idx]

dims = range(len(logvar_mean_sorted))

plt.figure(figsize=(14, 4))
plt.plot(dims, logvar_mean_sorted, linewidth=0.8, color='orange', label='mean')
plt.fill_between(dims, logvar_mean_sorted - logvar_std_sorted, logvar_mean_sorted + logvar_std_sorted, alpha=0.3, color='orange', label='±1 std')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel("Rank (sorted by mean logvar)")
plt.ylabel("logvar")
plt.title("logvar per latent dimension — sorted by mean")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
text_embeds_all = []
for i, batch in enumerate(dataloader):
    text_embs = batch["text_embedding"]
    text_embeds_all.append(text_embs)
    if i >= 99:  # 100 batches
        break

text_embeds_all = torch.cat(text_embeds_all, dim=0)
per_dim_std = text_embeds_all.std(dim=0)  # [768]
sort_idx = per_dim_std.argsort()
per_dim_std_sorted = per_dim_std[sort_idx]
plt.plot(dims, per_dim_std_sorted, linewidth=0.8, color='orange', label='mean')
plt.xlabel("Rank (sorted by std)")
plt.ylabel("std")
plt.title("Per-dimension std of text embeddings (sorted)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
layer_outputs = []


def make_hook(name):
    def hook(module, input, output):
        layer_outputs.append((name, output.detach()))

    return hook


handles = []
for name, module in model.encoder.named_modules():
    if isinstance(module, (nn.ReLU, nn.LayerNorm, nn.MaxPool2d, nn.Linear)):
        handles.append(module.register_forward_hook(make_hook(f"encoder.{name}")))

for name, module in model.fc_mu.named_modules():
    if isinstance(module, (nn.Linear,)):
        handles.append(module.register_forward_hook(make_hook(f"fc_mu.{name}")))

for name, module in model.fc_logvar.named_modules():
    if isinstance(module, (nn.Linear,)):
        handles.append(module.register_forward_hook(make_hook(f"fc_logvar.{name}")))

with torch.no_grad():
    imgs = next(iter(dataloader))["image"]
    intermediate = model.encoder(imgs)
    intermediate = torch.flatten(intermediate, start_dim=1)
    mu = model.fc_mu(intermediate)
    logvar = model.fc_logvar(intermediate)

for h in handles:
    h.remove()

# check dead ratio in each layer's output
for name, out in layer_outputs:
    flat = out.view(out.size(0), -1)
    dead_ratio = (flat.std(dim=0) < 0.01).float().mean().item()
    print(f"{name}: shape={list(out.shape)}, dead={dead_ratio:.1%}")